# Lab 06: Landmark Recognition Using SIFT Algorithm

[![Student ID](https://img.shields.io/badge/Student%20ID-FA23--BAI--039-blue)](https://github.com/asad-mashood)
[![Student](https://img.shields.io/badge/Student-Muhammad%20Asad%20Mashood-darkblue)](https://github.com/asad-mashood)
[![Repository](https://img.shields.io/badge/Repository-ComputerVision--LABS-teal)](https://github.com/asad-mashood/ComputerVision-LABS)
[![Course](https://img.shields.io/badge/Course-Computer%20Vision-blueviolet)](https://github.com/asad-mashood/ComputerVision-LABS)

**Student Name:** Muhammad Asad Mashood  
**Student ID:** FA23-BAI-039  
**Course:** Computer Vision  
**Assignment:** Lab 06 - Landmark Recognition Using SIFT  

---

## 1. Introduction and Objectives

The objective of this laboratory assignment is to develop an automated landmark recognition and geometric localization application using the Scale-Invariant Feature Transform (SIFT) algorithm, Brute-Force feature matching, and RANSAC homography estimation.

### Core Methodology
- **SIFT Feature Detection & Description:** Extraction of scale-space extrema and 128-dimensional gradient orientation histograms.
- **Brute-Force Matching & Lowe's Ratio Test:** k-NN matching ($k=2$) with distance ratio threshold $\tau = 0.75$.
- **Geometric Verification:** Planar homography estimation via RANSAC with geometric sanity checks (convexity, non-degenerate area).
- **Rejection Gate:** Automated rejection of unrecognized and non-database images.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from run_lab06 import task1_load_database, task2_extract_features, task3_process_query, task4_match_and_rank, task5_localize

print(f"OpenCV version: {cv2.__version__}")
sift = cv2.SIFT_create()
db = task1_load_database('data/landmarks')
db = task2_extract_features(db)
print(f"Successfully loaded and indexed {len(db)} landmark categories.")

## 2. Landmark Reference Database Gallery

Multi-view reference database containing 5 famous world landmarks:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (cat, items) in zip(axes, sorted(db.items())):
    rgb = cv2.cvtColor(items[0]['bgr'], cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    ax.set_title(cat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. SIFT Keypoints and Scale-Space Extrema

Keypoints detected via Difference-of-Gaussians (DoG) with orientation and scale attributes:

In [ ]:
sample_item = db['eiffel_tower'][0]
vis = cv2.drawKeypoints(sample_item['bgr'], sample_item['keypoints'][:500], None, color=(0, 255, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"SIFT Keypoints with Scale & Orientation (Sample 500 of {len(sample_item['keypoints'])})", fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()

## 4. Query Matching & RANSAC Localization

Evaluating test queries against the database:

In [ ]:
eiffel_ref = db['eiffel_tower'][0]['bgr']
h, w = eiffel_ref.shape[:2]
M = cv2.getRotationMatrix2D((w/2, h/2), 45, 0.85)
query_rot = cv2.warpAffine(eiffel_ref, M, (w, h), borderMode=cv2.BORDER_REFLECT)

gray_q, kps_q, des_q = task3_process_query(query_rot, sift)
is_rec, top, ranked, msg = task4_match_and_rank(des_q, db)
print(msg)

if is_rec and top['best_ref']:
    success, inliers, ann_img, loc_msg = task5_localize(
        query_rot, kps_q,
        top['best_ref']['bgr'], top['best_ref']['keypoints'],
        top['matches']
    )
    plt.figure(figsize=(9, 7))
    plt.imshow(cv2.cvtColor(ann_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Localized Landmark: {top['landmark'].title()} ({inliers} Inliers)", fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.show()

## 5. Experimental Benchmark Summary

| **Original Landmark Image** | `eiffel_tower` | `eiffel_tower` | `2,946` | `2,946` | `2,946` | **RECOGNIZED** | `1368.0 ms` |
| **Rotated Landmark Image (45 deg)** | `eiffel_tower` | `eiffel_tower` | `2,517` | `1,234` | `1,227` | **RECOGNIZED** | `1131.8 ms` |
| **Resized Landmark Image (0.5x Scale)** | `eiffel_tower` | `eiffel_tower` | `672` | `509` | `508` | **RECOGNIZED** | `372.5 ms` |
| **Reduced Brightness Image (0.35x)** | `eiffel_tower` | `eiffel_tower` | `760` | `718` | `717` | **RECOGNIZED** | `392.5 ms` |
| **Different Viewpoint Image** | `eiffel_tower` | `eiffel_tower` | `1,020` | `1,020` | `1,020` | **RECOGNIZED** | `431.6 ms` |
| **Non-Database Landmark (Big Ben)** | `unknown` | `Unknown / Rejected` | `1,984` | `22` | `0` | **REJECTED** | `895.3 ms` |